Basic Operations on Images 
==========================

Goal
----

Learn to:

-   Access pixel values and modify them
-   Access image properties
-   Set a Region of Interest (ROI)
-   Split and merge images

Almost all the operations in this section are mainly related to Numpy rather than OpenCV. A good
knowledge of Numpy is required to write better optimized code with OpenCV.

*( Examples will be shown in a Python terminal, since most of them are just single lines of code )*

Accessing and Modifying pixel values
------------------------------------

Let's load a color image first:

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt

img = cv.imread("../../assets/messi5.jpg")
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))

You can access a pixel value by its row and column coordinates. For BGR image, it returns an array
of Blue, Green, Red values. For grayscale image, just corresponding intensity is returned.

In [ ]:
px = img[100, 100]
px

In [ ]:
# accessing only blue pixel
blue = img[100, 100, 0]
blue

You can modify the pixel values the same way.

In [ ]:
img[100, 100] = [255, 255, 255]
img[100, 100]

**Warning**

Numpy is an optimized library for fast array calculations. So simply accessing each and every pixel
value and modifying it will be very slow and it is discouraged.

Accessing Image Properties
--------------------------

Image properties include number of rows, columns, and channels; type of image data; number of pixels; etc.

The shape of an image is accessed by img.shape. It returns a tuple of the number of rows, columns, and channels
(if the image is color):

In [ ]:
img.shape

> **Note:** If an image is grayscale, the tuple returned contains only the number of rows
and columns, so it is a good method to check whether the loaded image is grayscale or color.

Total number of pixels is accessed by `img.size`:

In [ ]:
img.size

Image datatype is obtained by \`img.dtype\`:

In [ ]:
img.dtype

> **Note:** img.dtype is very important while debugging because a large number of errors in OpenCV-Python
code are caused by invalid datatype.

Image ROI
---------

Sometimes, you will have to play with certain regions of images. For eye detection in images, first
face detection is done over the entire image. When a face is obtained, we select the face region alone
and search for eyes inside it instead of searching the whole image. It improves accuracy (because eyes
are always on faces :D ) and performance (because we search in a small area).

ROI is again obtained using Numpy indexing. Here I am selecting the ball and copying it to another
region in the image:

In [ ]:
import ipywidgets as widgets
from ipycanvas import Canvas

# Extract the ball region
ball = img[280:340, 330:390]
ball_h, ball_w = ball.shape[:2]

# Canvas for the interactive display
canvas = Canvas(width=img.shape[1], height=img.shape[0])

# Sliders to move the ROI destination
x_sl = widgets.IntSlider(value=100, min=0, max=img.shape[1] - ball_w, description="X")
y_sl = widgets.IntSlider(value=273, min=0, max=img.shape[0] - ball_h, description="Y")


def update(*_):
    out = img.copy()
    x, y = x_sl.value, y_sl.value
    out[y : y + ball_h, x : x + ball_w] = ball
    canvas.put_image_data(out, 0, 0)


for w in (x_sl, y_sl):
    w.observe(update, "value")
update()

display(widgets.VBox([widgets.HBox([x_sl, y_sl]), canvas]))

Splitting and Merging Image Channels
------------------------------------

Sometimes you will need to work separately on the B,G,R channels of an image. In this case, you need
to split the BGR image into single channels. In other cases, you may need to join these individual
channels to create a BGR image. You can do this simply by:

In [ ]:
import matplotlib.pyplot as plt

b, g, r = cv.split(img)
plt.subplot(131)
plt.imshow(b, cmap="Blues")
plt.subplot(132)
plt.imshow(g, cmap="Greens")
plt.subplot(133)
plt.imshow(r, cmap="Reds")
# hide the axis
for i in range(1, 4):
    plt.subplot(1, 3, i)
    plt.axis("off")
plt.show()

In [ ]:
merged_img = cv.merge((b, g, r))
plt.imshow(cv.cvtColor(merged_img, cv.COLOR_BGR2RGB))

Or
@code
>>> b = img[:,:,0]
@endcode
Suppose you want to set all the red pixels to zero - you do not need to split the channels first.
Numpy indexing is faster:

In [ ]:
img[:, :, 2] = 0
plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))

**Warning**

cv.split() is a costly operation (in terms of time). So use it only if necessary. Otherwise go
for Numpy indexing.

Making Borders for Images (Padding)
-----------------------------------

If you want to create a border around an image, something like a photo frame, you can use
**cv.copyMakeBorder()**. But it has more applications for convolution operation, zero
padding etc. This function takes following arguments:

-   **src** - input image
-   **top**, **bottom**, **left**, **right** - border width in number of pixels in corresponding
    directions

-   **borderType** - Flag defining what kind of border to be added. It can be following types:
    -   **cv.BORDER_CONSTANT** - Adds a constant colored border. The value should be given
        as next argument.
    -   **cv.BORDER_REFLECT** - Border will be mirror reflection of the border elements,
        like this : *fedcba|abcdefgh|hgfedcb*
    -   **cv.BORDER_REFLECT_101** or **cv.BORDER_DEFAULT** - Same as above, but with a
        slight change, like this : *gfedcb|abcdefgh|gfedcba*
    -   **cv.BORDER_REPLICATE** - Last element is replicated throughout, like this:
        *aaaaaa|abcdefgh|hhhhhhh*
    -   **cv.BORDER_WRAP** - Can't explain, it will look like this :
        *cdefgh|abcdefgh|abcdefg*

-   **value** - Color of border if border type is cv.BORDER_CONSTANT

Below is a sample code demonstrating all these border types for better understanding:

In [ ]:
import numpy as np

BLUE = [255, 0, 0]

# Load with alpha channel and replace transparency with white background
img1 = cv.imread("../../assets/opencv-logo.png", cv.IMREAD_UNCHANGED)
if img1.shape[2] == 4:
    alpha = img1[:, :, 3] / 255.0
    white_bg = np.full_like(img1[:, :, :3], 255, dtype=np.uint8)
    img1 = (
        img1[:, :, :3] * alpha[..., None] + white_bg * (1 - alpha[..., None])
    ).astype(np.uint8)

border_size = 100

replicate = cv.copyMakeBorder(
    img1, border_size, border_size, border_size, border_size, cv.BORDER_REPLICATE
)
reflect = cv.copyMakeBorder(
    img1, border_size, border_size, border_size, border_size, cv.BORDER_REFLECT
)
reflect101 = cv.copyMakeBorder(
    img1, border_size, border_size, border_size, border_size, cv.BORDER_REFLECT_101
)
wrap = cv.copyMakeBorder(
    img1, border_size, border_size, border_size, border_size, cv.BORDER_WRAP
)
constant = cv.copyMakeBorder(
    img1,
    border_size,
    border_size,
    border_size,
    border_size,
    cv.BORDER_CONSTANT,
    value=BLUE,
)

# Convert BGR (OpenCV) → RGB (matplotlib) and hide axes on all subplots
images = [img1, replicate, reflect, reflect101, wrap, constant]
titles = ["ORIGINAL", "REPLICATE", "REFLECT", "REFLECT_101", "WRAP", "CONSTANT"]

fig = plt.figure(facecolor="white")
for i, (im, title) in enumerate(zip(images, titles)):
    ax = plt.subplot(2, 3, i + 1)
    ax.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()